In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import os

import utils.prediction_utils as pred
from utils import dictionaries as dictionary
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR

from camp_tv_new.src.utils.logger import setup_logger


In [22]:
## FUNCTIONS
logger = setup_logger('logs/')

def prep_dataframe(df_bereik: pd.DataFrame) -> pd.DataFrame:


    df = df_bereik[["flight_id", "start_date", "end_date", "advertiser", "product", "daypart", "mock_campaign", "alcohol_kansspelen",
                    "npo1_grp", "npo2_grp", "npo3_grp", "regio_grp", "ster_grp",
                    "ster_1+"]].reset_index(drop=True)

    df = (df.pivot_table(index=["flight_id", "start_date", "end_date", 'advertiser', 'product', "mock_campaign", "alcohol_kansspelen"],
                         columns='daypart', fill_value=0.0,
                         values=['npo1_grp', 'npo2_grp', 'npo3_grp',
                                 'ster_grp', 'ster_1+', 'regio_grp'], aggfunc='sum').reset_index())

    df.columns = ["_".join(col) if col[1] else col[0] for col in df.columns.to_flat_index()]

    # Convert start_date and end_date to datetime objects and calculate required values
    df['start_date'] = pd.to_datetime(df['start_date'], format="%d%m%y", errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], format="%d%m%y", errors='coerce')

    # Apply function to create extra columns
    df['timef_start_month'] = df['start_date'].dt.month
    df['timef_end_month'] = df['end_date'].dt.month
    df['timef_start_year'] = df['start_date'].dt.year
    df['timef_end_year'] = df['end_date'].dt.year
    df['timef_start_week'] = df['start_date'].dt.strftime("%V").astype(int)
    df['timef_end_week'] = df['end_date'].dt.strftime("%V").astype(int)
    df['timef_start_day'] = df['start_date'].dt.day
    df['timef_end_day'] = df['end_date'].dt.day
    df['timef_duration_days'] = df.apply(lambda row: (row['end_date'] - row['start_date']).days + 1, axis=1)

    # Calculate total overlap with olympics and WK/EK
    df['timef_olympic_period_overlap'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.olympic_period_list))
    df['timef_football_period_overlap'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.football_wk_ek_list))

    return df

def create_preprocessor(numerical_cols, categorical_cols, boolean_cols):
    """
    Create a ColumnTransformer with specified transformations for numerical, categorical, and boolean columns.

    Args:
    numerical_cols (list of str): List of numerical column names.
    categorical_cols (list of str): List of categorical column names.
    boolean_cols (list of str): List of boolean column names.

    Returns:
    ColumnTransformer: A preprocessor with the specified transformations.
    """

    # Preprocessing for numerical data: impute missing values and scale
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())])

    # Preprocessing for categorical data: impute missing values and one-hot encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    # Preprocessing for boolean data: impute missing values (if any)
    boolean_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent'))])

    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols),
            # ('bool', boolean_transformer, boolean_cols)
            ])

    return preprocessor

def grid_search_models(X_train, y_train, X_test, y_test, models, preprocessor):
    # Initialize variables to store the best model and its score
    best_model = None
    best_score = float('inf')
    best_params = None
    results = []

    # Perform GridSearchCV for each model
    for model_info in models:
        # Create a pipeline with the preprocessor and the model
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model_info['model'])
        ])

        # Create GridSearchCV
        grid_search = GridSearchCV(pipeline, model_info['params'], cv=5, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True)
        grid_search.fit(X_train, y_train)
        results.append(pd.DataFrame(grid_search.cv_results_))

        # Get the best score and parameters
        if grid_search.best_score_ < best_score:
            best_score = grid_search.best_score_
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_

    # Evaluate the best model on the test set
    y_pred = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)

    logger.info(f"Best Model: {best_model}")
    logger.info(f"Best Parameters: {best_params}")
    logger.info(f"Test MSE: {test_mse}")

    return results, best_model, best_params, test_mse

def query_to_df(query):
    engine = create_sql_alchemy_engine()
    df_query = pd.read_sql_query(query, engine)

    return df_query

def create_sql_alchemy_engine():
    from sqlalchemy import create_engine
    from urllib.parse import quote_plus
    """
    Creates and returns a SQLAlchemy engine for connecting to a SQL Server database.
    """
    quoted = quote_plus(
        "Driver={ODBC+Driver+17+for+SQL+Server};"
        "Server=tcp:stercampsql.database.windows.net;"
        "Database=SterCampNew;"
        "uid=stercampadmin;"
        f"pwd={os.getenv('SQLDB')}"
    )
    alchemy_con = "mssql+pyodbc:///?odbc_connect={}".format(quoted)
    engine = create_engine(alchemy_con)

    return engine

In [26]:
## VARIABLES TO TWEAK

TEST_SET_DATE = pd.to_datetime('2024-05-01')
CUT_OFF_DATE = '2023-09-01'
TARGET_GROUP = '25-67'

######################################################################################################

query = f'''
    SELECT b.flight_id,
    f.advertiser,
    f.product,
    f.start_date,
    f.end_date,
    b.target_group,
    b.campaign_duration,
    b.daypart,
    f.mock_campaign,
    f.alcohol_kansspelen,
    b.total_grp,
    b.ster_grp,
    b.regio_grp,
    b.npo1_grp,
    b.npo2_grp,
    b.npo3_grp,
    b.[ster_1+]
    FROM [dbo].[tv.bereik] b
    LEFT JOIN [dbo].[tv.flights_ster] f ON b.flight_id = f.flight_id
    where b.ster_grp > 0 and target_group = '{TARGET_GROUP}'
    and start_date >= '{CUT_OFF_DATE}'
    order by start_date
'''

try:
    df = query_to_df(query)
except Exception as e:
    print(f"Error executing query: {str(e)}")

# Prep df
df = prep_dataframe(df)

# Ensure the data is sorted by the start date
df = df.sort_values(by='start_date')

# Split the data into training and test sets based on the cutoff date
train_data = df[df['start_date'] < TEST_SET_DATE]
test_data = df[df['start_date'] >= TEST_SET_DATE]

features = ['npo1_grp_daynight', 'npo1_grp_primetime', 'npo1_grp_total',
            'npo2_grp_daynight', 'npo2_grp_primetime', 'npo2_grp_total',
            'npo3_grp_daynight', 'npo3_grp_primetime', 'npo3_grp_total',
            'regio_grp_daynight', 'regio_grp_primetime', 'regio_grp_total',
            'timef_start_month', 'timef_end_month', 'timef_start_year', 'timef_end_year',
            'timef_start_week', 'timef_end_week', 'timef_start_day', 'timef_end_day',
            'timef_duration_days', 'timef_olympic_period_overlap', 'timef_football_period_overlap',
            'alcohol_kansspelen']

target = 'ster_1+_total'

# Separate features and target for train and test sets
X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

# Create the preprocessor. (nu niet echt nodig maar wellicht wel later in de pipeline)
categorical_cols = []
boolean_cols = ['alcohol_kansspelen']
numerical_cols = [col for col in features if col not in categorical_cols + boolean_cols]
for col in boolean_cols:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)
preprocessor = create_preprocessor(numerical_cols, categorical_cols, boolean_cols)

df.head(2)

C:\Users\jordy.schep\AppData\Local\Temp\ipykernel_23336\2262253678.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[col] = X_train[col].astype(int)
C:\Users\jordy.schep\AppData\Local\Temp\ipykernel_23336\2262253678.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = X_test[col].astype(int)


,flight_id,start_date,end_date,advertiser,product,mock_campaign,alcohol_kansspelen,npo1_grp_daynight,npo1_grp_primetime,npo1_grp_total,...,timef_end_month,timef_start_year,timef_end_year,timef_start_week,timef_end_week,timef_start_day,timef_end_day,timef_duration_days,timef_olympic_period_overlap,timef_football_period_overlap
326,2bd1aca8-21eb-0f4e-2751-61a993ecd93d,2023-09-01,2023-10-31,Leidse onderwijsinstellingen,Loi,False,False,0.0,0.00,0.00,...,10,2023,2023,35,44,1,31,61,0,0
1499,c52d6561-69a7-dd84-a887-325bf9e3ccfe,2023-09-01,2023-10-31,Pearle,Pearle,False,False,34.1,66.89,100.98,...,10,2023,2023,35,44,1,31,61,0,0


In [27]:
models = [
    {
        'name': 'XGBoost',
        'model': XGBRegressor(objective='reg:squarederror', random_state=42),
        'params': {
            'model__n_estimators': [50, 100, 500],
            'model__learning_rate': [0.01, 0.05, 0.1],
            'model__max_depth': [3, 5, 10, 12],
            'model__gamma': [0, 0.2, 0.4],
            'model__subsample': [0.6, 1.0],
            'model__colsample_bytree': [0.6, 1.0],
            'model__min_child_weight': [1, 3, 5],
            'model__reg_alpha': [0, 0.01, 0.1],
            'model__reg_lambda': [1, 2]
        }
    },
    # {
    #     'name': 'GradientBoosting',
    #     'model': GradientBoostingRegressor(),
    #     'params': {
    #         'model__n_estimators': [50, 100, 300, 500, 1000],
    #         'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
    #         'model__max_depth': [3, 5, 7, 10, 12],
    #         'model__min_samples_split': [2, 5, 10],
    #         'model__min_samples_leaf': [1, 2, 4],
    #         'model__subsample': [0.6, 0.8, 1.0]
    #     }
    # },
    # {
    #     'name': 'SVR',
    #     'model': SVR(),
    #     'params': {
    #         'model__C': [0.1, 1, 10, 100],
    #         'model__gamma': ['scale', 'auto'],
    #         'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    #         'model__degree': [2, 3, 4],
    #         'model__epsilon': [0.1, 0.2, 0.5, 1.0]
    #     }
    # }
]

In [28]:
# Run the grid search function
results, best_model, best_params, test_mse = grid_search_models(X_train, y_train, X_test, y_test, models, preprocessor)

18:29:48 - INFO - [      mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0          0.165463      0.016763         0.016057        0.003106   
1          0.120177      0.016837         0.010814        0.002909   
2          0.203228      0.115616         0.010031        0.002233   
3          0.127681      0.019182         0.010841        0.004639   
4          0.131485      0.008380         0.009669        0.001726   
...             ...           ...              ...             ...   
7771       2.495461      0.497496         0.024437        0.009747   
7772       3.382973      0.314100         0.030346        0.013673   
7773       2.975576      0.122660         0.025058        0.004627   
7774       3.086132      0.352234         0.021613        0.004542   
7775       2.111772      0.259484         0.026468        0.023822   

     param_model__colsample_bytree param_model__gamma  \
0                              0.6                  0   
1                         

[      mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0          0.165463      0.016763         0.016057        0.003106   
1          0.120177      0.016837         0.010814        0.002909   
2          0.203228      0.115616         0.010031        0.002233   
3          0.127681      0.019182         0.010841        0.004639   
4          0.131485      0.008380         0.009669        0.001726   
...             ...           ...              ...             ...   
7771       2.495461      0.497496         0.024437        0.009747   
7772       3.382973      0.314100         0.030346        0.013673   
7773       2.975576      0.122660         0.025058        0.004627   
7774       3.086132      0.352234         0.021613        0.004542   
7775       2.111772      0.259484         0.026468        0.023822   

     param_model__colsample_bytree param_model__gamma  \
0                              0.6                  0   
1                              0.6          

In [14]:
from sklearn.model_selection import RandomizedSearchCV

# Define the XGBRegressor model with early stopping
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# Define the refined parameter grid for RandomizedSearchCV
param_grid = {
    'n_estimators': [400, 450, 500, 550, 600],
    'learning_rate': [0.005, 0.01, 0.015, 0.02],
    'max_depth': [6, 7, 8],
    'gamma': [0.2, 0.3, 0.4],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.75, 0.8, 0.85],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

# Set up RandomizedSearchCV with 10-fold cross-validation
random_search = RandomizedSearchCV(estimator=xgb, param_distributions=param_grid, n_iter=100, cv=10, n_jobs=-1, scoring='neg_mean_squared_error', random_state=42)


# Fit the RandomizedSearchCV to the training data with early stopping
random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

# Print the best parameters and best score
print("Best Parameters:", random_search.best_params_)
print("Best Score:", -random_search.best_score_)

[0]	validation_0-rmse:15.58717
[1]	validation_0-rmse:15.39777
[2]	validation_0-rmse:15.21307
[3]	validation_0-rmse:15.03016
[4]	validation_0-rmse:14.84665
[5]	validation_0-rmse:14.67356
[6]	validation_0-rmse:14.51365
[7]	validation_0-rmse:14.34301
[8]	validation_0-rmse:14.17552
[9]	validation_0-rmse:14.01357
[10]	validation_0-rmse:13.84837
[11]	validation_0-rmse:13.67922
[12]	validation_0-rmse:13.52811
[13]	validation_0-rmse:13.37504
[14]	validation_0-rmse:13.22747
[15]	validation_0-rmse:13.07566
[16]	validation_0-rmse:12.93841
[17]	validation_0-rmse:12.79909
[18]	validation_0-rmse:12.65514
[19]	validation_0-rmse:12.52037
[20]	validation_0-rmse:12.37697
[21]	validation_0-rmse:12.24280
[22]	validation_0-rmse:12.10701
[23]	validation_0-rmse:11.97602
[24]	validation_0-rmse:11.90316
[25]	validation_0-rmse:11.77748
[26]	validation_0-rmse:11.64929
[27]	validation_0-rmse:11.52631
[28]	validation_0-rmse:11.40638
[29]	validation_0-rmse:11.29305
[30]	validation_0-rmse:11.17887
[31]	validation_0-